In [ ]:
!git clone -q https://github.com/shemshallah/nanlmv9.git
%cd nanlmv9
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!mkdir -p data
!curl -sSL -o data/wt2_train.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt
!python3 experiments/mamba_moe/tokenizer_bpe.py --vocab-texts data/wt2_train.txt --train-texts data/wt2_train.txt --out data/tok32k --vocab-size 32000

In [ ]:
!curl -L --retry 2 -o cm_valid_full.bin https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-full/cm_valid_full.bin
!curl -L --retry 2 -o win_valid_full.bin https://github.com/shemshallah/nanlmv9/releases/download/cm-prior-full/win_valid_full.bin
!ls -lh cm_valid_full.bin win_valid_full.bin


In [ ]:
!python3 experiments/mamba_moe/train_byte_residual.py \
  --data-dir data/tok32k \
  --cm-dump cm_valid_full.bin \
  --window win_valid_full.bin \
  --out runs/byte_residual --steps 10000 --seq 32 \
  --d-model 512 --layers 1 --experts 32 --lr 3e-4 \
  --device cuda --amp --max-seconds 600 \
  --result-json runs/byte_residual/result.json

In [ ]:
import json; print(open('runs/byte_residual/result.json').read())